# 🎵 MusicAI Studio — live test on a free Google Colab GPU

Run every cell top to bottom (**Runtime → Run all**). The last cell prints a public URL — open it on your phone or laptop and generate real AI songs.

**Before running:** go to **Runtime → Change runtime type → T4 GPU** (free tier) or **A100/L4** (Colab Pro, much faster).

In [ ]:
# 1) Check which GPU you got
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Get the app + install everything (~5-10 minutes)
# NOTE: a red "dependency resolver" warning about gcsfs/fsspec at the end
# is SAFE TO IGNORE — gcsfs is a Colab pre-install our app never uses.
# As long as you see "Building wheel for ace-step ... done", you're good.
!git clone https://github.com/ppratik2026/Musicai.git
%cd Musicai
!pip install -q -r requirements.txt
!pip install -q -r requirements-models.txt

In [ ]:
# 3) Tunnel tool for a public URL (no signup needed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 4) Start MusicAI + print your public URL
import os, re, subprocess, time

import torch

# GPU-specific setup:
# - A100/L4 (Ampere+): bfloat16, everything on GPU -> fastest
# - T4 (free tier): float16 diffusion + float32 audio decode + CPU offload
env = dict(os.environ)
if torch.cuda.is_bf16_supported():
    env["MUSICAI_ACE_DTYPE"] = "bfloat16"
else:
    env["MUSICAI_ACE_DTYPE"] = "float16"
    env["MUSICAI_ACE_CPU_OFFLOAD"] = "1"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("ACE-Step dtype:", env["MUSICAI_ACE_DTYPE"],
      "| cpu_offload:", env.get("MUSICAI_ACE_CPU_OFFLOAD", "0"))

server = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"], env=env
)
time.sleep(8)

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stderr=subprocess.PIPE, text=True,
)
for line in tunnel.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        print("\n" + "=" * 60)
        print("🎵 YOUR LIVE APP:", m.group(0))
        print("=" * 60)
        print("\nOpen the URL, pick the ACE-Step engine, and generate!")
        print("First generation downloads ~7 GB of model weights — be patient.")
        break

## Tips

- **Vocals need lyrics!** Fill the lyrics box (use `[verse]` / `[chorus]` sections) — an empty lyrics box gives you an instrumental.
- **Full-length songs (3–4 min):** pick the **ACE-Step** engine and set the duration slider to 180–240 s.
- **First song is slow** (checkpoint download + model load). After that it's much faster.
- On the free **T4**, a 3-minute ACE-Step song can take several minutes to render (float16 + CPU offload mode). A100/L4 (Colab Pro) is several times faster.
- Keep this tab open — closing Colab stops your server. Download the WAV/MP3 of anything you like before the session ends.
- ACE-Step is Apache-2.0 and watermark-free — your releases are yours. See `docs/DISTRIBUTION.md` in the repo.